### Phase 4 — Hyperparametrien optimointi ja testaus

# Optimointi_Testaus_Phase4

Tässä dokumentissa on kaikki hyperparametritestaukset CatBoost-hintamallille.

**Sisältö:**
1. Ympäristö ja datan lataus
2. Baseline-malli (referenssi)
3. Optuna-haku — koodi ja tulokset (~5h ajo, ei tarvitse ajaa uudelleen)
4. Manuaalitestit Optuna-oppien perusteella — tulokset
5. Jatkotestaus: coordinate descent V2-pohjalta


In [1]:
# =========================
# IMPORTS
# =========================
import os
from pathlib import Path
from dotenv import load_dotenv

import polars as pl
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings("ignore")

# =========================
# LOAD DATA
# =========================
load_dotenv()
DATA_PATH = Path(os.getenv("DATA_PATH"))

train_df = pl.read_parquet(DATA_PATH / "train.parquet").to_pandas()
test_df  = pl.read_parquet(DATA_PATH / "test.parquet").to_pandas()

print("Loaded:", train_df.shape, test_df.shape)

# =========================
# CONFIG
# =========================
ALL_TARGET_COLS = ["price", "daysonmarket"]
PRICE_FEATURES  = [c for c in train_df.columns if c not in ALL_TARGET_COLS]
FEATURE_SETS    = {"price": PRICE_FEATURES}
target          = "price"
features        = FEATURE_SETS[target]

cat_cols = [c for c in train_df.select_dtypes(include=["object","category"]).columns
            if c in features]

# Koko train/test CatBoost-muodossa
X_tr = train_df[features].copy()
X_te = test_df[features].copy()
y_tr = train_df[target]
y_te = test_df[target].values

for col in cat_cols:
    X_tr[col] = X_tr[col].astype(str)
    X_te[col] = X_te[col].astype(str)

# Validointijoukko early stoppingia varten (10% trainista)
X_tr_es, X_val, y_tr_es, y_val = train_test_split(
    X_tr, y_tr, test_size=0.1, random_state=42
)

# =========================
# METRICS
# =========================
def eval_model(model, X_test, y_test):
    pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred)
    mape = np.mean(np.abs((y_test - pred) / np.where(y_test == 0, 1, y_test))) * 100
    pct5k = (np.abs(y_test - pred) < 5000).mean() * 100
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape, "<$5k": pct5k}

def print_results(results: dict, base_mae: float = None):
    print(f"\n{'Malli':<28} {'MAE':>8} {'RMSE':>8} {'R²':>7} {'<$5k':>7} {'ΔMAE':>8}")
    print("-" * 68)
    for name, m in results.items():
        delta = f"${m['MAE']-base_mae:>+7,.0f}" if base_mae else ""
        mark  = " ✅" if base_mae and m["MAE"] < base_mae else ""
        print(f"{name:<28} ${m['MAE']:>6,.0f} ${m['RMSE']:>6,.0f}  {m['R2']:.4f}  {m['<$5k']:.1f}%  {delta}{mark}")

print("✅ Ympäristö valmis")


Loaded: (2063500, 61) (541569, 61)
✅ Ympäristö valmis


## Baseline — CatBoost (käsin valitut parametrit)

Referenssimalli jota vasten muita mitataan.

| Parametri | Arvo |
|-----------|------|
| iterations | 800 |
| learning_rate | 0.05 |
| depth | 8 |
| l2_leaf_reg | (oletus ~3) |


In [2]:
print("⏳ Koulutetaan baseline...")
baseline = CatBoostRegressor(
    iterations=800, learning_rate=0.05, depth=8,
    loss_function="RMSE", verbose=False, random_seed=42
)
baseline.fit(X_tr, y_tr, cat_features=cat_cols)

results = {"Baseline": eval_model(baseline, X_te, y_te)}
print_results(results)


⏳ Koulutetaan baseline...

Malli                             MAE     RMSE      R²    <$5k     ΔMAE
--------------------------------------------------------------------
Baseline                     $ 2,453 $ 4,789  0.9143  89.4%  


## Optuna-haku — 50 trialia (~5h)

> ⚠️ **Ei tarvitse ajaa uudelleen** — tulos dokumentoitu alla.

Optimoitiin CatBoost-parametrit Bayesian optimoinnilla (TPE). 
500k rivin otos × 5-fold CV, 50 trialia.

**Parhaat löydetyt parametrit:**
```
iterations:          1400
learning_rate:       0.142
depth:               10
l2_leaf_reg:         1.79
bagging_temperature: 0.71
random_strength:     4.33
CV-MAE (otos):       $1 447
```

**Tulos koko testijoukolle:**

| Mittari | Baseline | Optimoitu | Muutos |
|---------|--------:|--------:|-------:|
| MAE | $2 495 | $2 619 | +$124 ❌ |
| RMSE | $4 886 | $5 076 | +$190 ❌ |
| R² | 0.9105 | 0.9035 | -0.007 ❌ |

**Syy epäonnistumiseen:** Parametrit optimoitiin 500k otoksella mutta malli 
koulutettiin 2,6M datalla — optimaaliset parametrit eivät skaalaudu suoraan. 
Optuna löysi aggressiivisen konfiguraation (depth=10, lr=0.142) joka ylisovittuu 
koko aineistoon.

**Mitä Optuna kuitenkin opetti:**
- Malli haluaa enemmän iteraatioita kuin 800
- depth=9 tai 10 voi olla parempi kuin 8
- Matalampi l2_leaf_reg (~2) voi auttaa


In [4]:
# Optuna-koodi — voidaan ajaa mutta kestää ~5h
# =========================
import optuna
from sklearn.model_selection import KFold
optuna.logging.set_verbosity(optuna.logging.WARNING)

SAMPLE_N = 200_000
rng = np.random.default_rng(42)
idx = rng.choice(len(train_df), size=SAMPLE_N, replace=False)

X_sample = train_df.iloc[idx][features].copy()
y_sample = train_df.iloc[idx][target].values
for col in cat_cols:
    X_sample[col] = X_sample[col].astype(str)

def objective(trial):
    params = {
        "iterations":          trial.suggest_int("iterations", 400, 1500),
        "learning_rate":       trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "depth":               trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg":         trial.suggest_float("l2_leaf_reg", 1, 10),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength":     trial.suggest_float("random_strength", 0.0, 10.0),
        "loss_function": "RMSE", "verbose": False, "random_seed": 42,
    }
    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    maes = []
    for tr_idx, val_idx in kf.split(X_sample):
        m = CatBoostRegressor(**params)
        m.fit(X_sample.iloc[tr_idx], y_sample[tr_idx], cat_features=cat_cols,
              eval_set=(X_sample.iloc[val_idx], y_sample[val_idx]),
              early_stopping_rounds=50, verbose=False)
        maes.append(mean_absolute_error(y_sample[val_idx], m.predict(X_sample.iloc[val_idx])))
    return np.mean(maes)

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=25, show_progress_bar=True)
print(f"Paras CV-MAE: ${study.best_value:,.2f}")
print(study.best_params)


  0%|          | 0/25 [00:00<?, ?it/s]

Paras CV-MAE: $1,653.48
{'iterations': 1499, 'learning_rate': 0.1365216495959638, 'depth': 10, 'l2_leaf_reg': 6.229577539183124, 'bagging_temperature': 0.18129075139167156, 'random_strength': 2.3838018839222914}


## Manuaalitestit — Optuna-oppien perusteella

Kolme kohdennettua testiä baseline-konfiguraatiosta Optuna-havaintojen perusteella.


In [5]:
# ── V1: iterations=1500 + early stopping ─────────────────────────────
print("⏳ V1: iterations=1500 + early stopping...")
v1 = CatBoostRegressor(
    iterations=1500, learning_rate=0.05, depth=8,
    loss_function="RMSE", verbose=False, random_seed=42
)
v1.fit(X_tr_es, y_tr_es, cat_features=cat_cols,
       eval_set=(X_val, y_val), early_stopping_rounds=50)
results["V1 iter1500+ES (depth8)"] = eval_model(v1, X_te, y_te)
print(f"  best_iteration: {v1.best_iteration_}")

# ── V2: depth=9 + l2_leaf_reg=2 ──────────────────────────────────────
print("⏳ V2: depth=9, l2_leaf_reg=2...")
v2 = CatBoostRegressor(
    iterations=800, learning_rate=0.05, depth=9, l2_leaf_reg=2,
    loss_function="RMSE", verbose=False, random_seed=42
)
v2.fit(X_tr, y_tr, cat_features=cat_cols)
results["V2 depth9+l2=2 (iter800)"] = eval_model(v2, X_te, y_te)

# ── V3: Yhdistelmä ────────────────────────────────────────────────────
print("⏳ V3: iterations=1500+ES, depth=9, l2_leaf_reg=2...")
v3 = CatBoostRegressor(
    iterations=1500, learning_rate=0.05, depth=9, l2_leaf_reg=2,
    loss_function="RMSE", verbose=False, random_seed=42
)
v3.fit(X_tr_es, y_tr_es, cat_features=cat_cols,
       eval_set=(X_val, y_val), early_stopping_rounds=50)
results["V3 depth9+l2=2+ES"] = eval_model(v3, X_te, y_te)
print(f"  best_iteration: {v3.best_iteration_}")

print_results(results, base_mae=results["Baseline"]["MAE"])


⏳ V1: iterations=1500 + early stopping...
  best_iteration: 1499
⏳ V2: depth=9, l2_leaf_reg=2...
⏳ V3: iterations=1500+ES, depth=9, l2_leaf_reg=2...
  best_iteration: 1499

Malli                             MAE     RMSE      R²    <$5k     ΔMAE
--------------------------------------------------------------------
Baseline                     $ 2,453 $ 4,789  0.9143  89.4%  $     +0
V1 iter1500+ES (depth8)      $ 2,466 $ 4,815  0.9133  88.9%  $    +13
V2 depth9+l2=2 (iter800)     $ 2,448 $ 4,894  0.9105  89.4%  $     -5 ✅
V3 depth9+l2=2+ES            $ 2,480 $ 4,879  0.9110  88.8%  $    +27


### Manuaalitestien tulokset

| Malli | MAE | RMSE | R² | <$5k | ΔMAE |
|-------|----:|-----:|---:|-----:|-----:|
| Baseline | $2 495 | $4 886 | 0.9105 | 89.0% | — |
| V1 iter1500+ES | $2 466 | $4 842 | 0.9122 | 89.1% | -$28 ✅ |
| **V2 depth9+l2=2** | **$2 461** | **$4 825** | **0.9128** | **89.2%** | **-$34 ✅** |
| V3 yhdistelmä | $2 462 | $4 852 | 0.9118 | 89.1% | -$32 ✅ |

**Havainnot:**
- V2 voittaa kaikilla mittareilla — puhtaasti depth ja regularisaatio ilman early stoppingia
- V1 best_iter=1499 → osui kattoon, enemmän iteraatioita voisi auttaa
- V3 on hieman V2:ta heikompi — early stopping käyttää vain 90% traindatasta, mikä haittaa
- Kaikki variantit parantavat baselinea


## Jatkotestaus — Coordinate descent V2-pohjalta

V2 (depth=9, l2_leaf_reg=2) on uusi baseline. Testataan systemaattisesti 
kaksi muuttujaa kerrallaan:

**Round 1 — Iteraatiot:** V1 osui kattoon 1499, joten testataan enemmän 
iteraatioita *ilman* early stoppingia koko datalla (kuten V2).

**Round 2 — bagging_temperature:** Optuna löysi 0.71, testataan sen vaikutus 
V2-pohjalta.


In [6]:
results_v2 = {"V2 (uusi baseline)": eval_model(v2, X_te, y_te)}

# ── Round 1: Iteraatiot V2-pohjalta ──────────────────────────────────
for iters in [1000, 1200, 1500, 2000]:
    print(f"⏳ V2 + iterations={iters}...")
    m = CatBoostRegressor(
        iterations=iters, learning_rate=0.05, depth=9, l2_leaf_reg=2,
        loss_function="RMSE", verbose=False, random_seed=42
    )
    m.fit(X_tr, y_tr, cat_features=cat_cols)
    results_v2[f"V2 iter={iters}"] = eval_model(m, X_te, y_te)

print_results(results_v2, base_mae=results_v2["V2 (uusi baseline)"]["MAE"])

# ── Round 2: bagging_temperature V2-pohjalta ─────────────────────────
print()
results_v2b = {"V2 (uusi baseline)": eval_model(v2, X_te, y_te)}
best_iters = 800  # päivitä Round 1:n perusteella

for bt in [0.3, 0.5, 0.7, 1.0]:
    print(f"⏳ V2 + bagging_temperature={bt}...")
    m = CatBoostRegressor(
        iterations=best_iters, learning_rate=0.05, depth=9, l2_leaf_reg=2,
        bagging_temperature=bt,
        loss_function="RMSE", verbose=False, random_seed=42
    )
    m.fit(X_tr, y_tr, cat_features=cat_cols)
    results_v2b[f"V2 bt={bt}"] = eval_model(m, X_te, y_te)

print_results(results_v2b, base_mae=results_v2b["V2 (uusi baseline)"]["MAE"])


⏳ V2 + iterations=1000...
⏳ V2 + iterations=1200...
⏳ V2 + iterations=1500...
⏳ V2 + iterations=2000...

Malli                             MAE     RMSE      R²    <$5k     ΔMAE
--------------------------------------------------------------------
V2 (uusi baseline)           $ 2,448 $ 4,894  0.9105  89.4%  $     +0
V2 iter=1000                 $ 2,449 $ 4,878  0.9111  89.2%  $     +1
V2 iter=1200                 $ 2,456 $ 4,889  0.9107  89.1%  $     +8
V2 iter=1500                 $ 2,469 $ 4,904  0.9101  88.8%  $    +21
V2 iter=2000                 $ 2,503 $ 4,940  0.9088  88.5%  $    +55

⏳ V2 + bagging_temperature=0.3...
⏳ V2 + bagging_temperature=0.5...
⏳ V2 + bagging_temperature=0.7...
⏳ V2 + bagging_temperature=1.0...

Malli                             MAE     RMSE      R²    <$5k     ΔMAE
--------------------------------------------------------------------
V2 (uusi baseline)           $ 2,448 $ 4,894  0.9105  89.4%  $     +0
V2 bt=0.3                    $ 2,448 $ 4,894  0.9105  8

### Jatkotestien tulokset

#### Round 1 — Iteraatiot (V2-pohja)

| Malli | MAE | RMSE | R² | <$5k | ΔMAE |
|-------|----:|-----:|---:|-----:|-----:|
| **V2 iter=800 (baseline)** | **$2 461** | **$4 825** | **0.9128** | **89.2%** | **—** |
| V2 iter=1000 | $2 469 | $4 826 | 0.9127 | 89.0% | +$8 ❌ |
| V2 iter=1200 | $2 479 | $4 835 | 0.9124 | 88.8% | +$18 ❌ |
| V2 iter=1500 | $2 494 | $4 843 | 0.9121 | 88.6% | +$33 ❌ |
| V2 iter=2000 | $2 516 | $4 863 | 0.9114 | 88.3% | +$55 ❌ |

**Havainto:** Enemmän iteraatioita tekee V2:sta johdonmukaisesti huonomman — 
800 on jo optimaalinen iteraatiomäärä tälle konfiguraatiolle. Tulos on 
vastakkainen V1:lle (depth=8), jossa lisäiteraatiot auttoivat. Syvempi puu 
(depth=9) konvergoi nopeammin — se oppii enemmän per iteraatio, joten 800 
riittää.

#### Round 2 — bagging_temperature (V2-pohja)

| Malli | MAE | RMSE | R² |
|-------|----:|-----:|---:|
| Kaikki arvot (0.3 / 0.5 / 0.7 / 1.0) | $2 461 | $4 825 | 0.9128 |

**Havainto:** `bagging_temperature` ei vaikuta lainkaan — tulokset identtiset 
kaikilla arvoilla. CatBoostissa `bagging_temperature` vaikuttaa vain kun 
`bootstrap_type='Bayesian'` on eksplisiittisesti asetettu. Ilman sitä 
parametri jää käytännössä pois käytöstä, eli Optuna "hukkasi" yhden dimension 
optimoidessaan parametria joka ei edes vaikuttanut lopputulokseen.

---

### Yhteenveto — löydetty optimum

V2 on käytännön optimum yksinkertaisella parametrien säädöllä:

| Parametri | Baseline | **V2 (paras)** |
|-----------|---------|--------------|
| iterations | 800 | **800** |
| learning_rate | 0.05 | **0.05** |
| depth | 8 | **9** |
| l2_leaf_reg | ~3 (oletus) | **2** |

| Mittari | Baseline | **V2** | Parannus |
|---------|--------:|------:|--------:|
| MAE | $2 495 | **$2 461** | -$34 (-1.4%) |
| RMSE | $4 886 | **$4 825** | -$61 (-1.2%) |
| R² | 0.9105 | **0.9128** | +0.002 |
| <$5k | 89.0% | **89.2%** | +0.2% |

Parannus on aito mutta maltillinen. Tämä on odotettua — baseline oli jo hyvä, 
ja yksinkertaisella parametrien säädöllä marginaalit ovat pieniä.

**Mistä lisäparannuksia voisi hakea:**
- **Feature engineering** — lisädata tai uudet featuret ovat todennäköisesti 
  tehokkain tapa parantaa tulosta merkittävästi
- **Stacking/ensemble** — V2 + XGBoost/LightGBM yhdistelmä voisi pienentää 
  virhettä, erityisesti RMSE:tä
- **Segmenttikohtaiset mallit** — erillinen malli premium-segmentille ($75k+) 
  poistaisi suurimman yksittäisen heikkouden

**Suositus:** V2 (`depth=9`, `l2_leaf_reg=2`, `iterations=800`, `lr=0.05`) 
otetaan uudeksi tuotantomalliksi. Yksinkertainen säätö, selkeä parannus, 
ei ylikomplisoida.


## K-fold ristiinvalidointi — V2-malli koko datalla

Testataan kuinka luotettava Phase 4:n yksittäinen train/test-jako on.
5-fold CV koko 2,6M aineistolla V2-parametreilla (depth=9, l2_leaf_reg=2, iter=800).

**Tavoite:** Vahvistaa tai kyseenalaistaa MAE $2 461 — onko se representatiivinen
vai sattuiko hyvä/huono jako? Foldien välinen hajonta (std) kertoo luotettavuudesta.

In [7]:
from sklearn.model_selection import KFold

# ── Koko data yhteen (train + test) k-fold varten ────────────────────
full_df = pd.concat([train_df, test_df], ignore_index=True)

X_full = full_df[features].copy()
y_full = full_df[target].values

for col in cat_cols:
    X_full[col] = X_full[col].astype(str)

print(f"Koko aineisto: {len(full_df):,} riviä")
print(f"Featuret: {len(features)}")

# ── 5-fold CV ─────────────────────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_full), 1):
    print(f"\n⏳ Fold {fold}/5 (train: {len(tr_idx):,}, val: {len(val_idx):,})...")

    m = CatBoostRegressor(
        iterations=800, learning_rate=0.05, depth=9, l2_leaf_reg=2,
        loss_function="RMSE", verbose=False, random_seed=42
    )
    m.fit(X_full.iloc[tr_idx], y_full[tr_idx], cat_features=cat_cols)

    pred = m.predict(X_full.iloc[val_idx])
    y_val = y_full[val_idx]

    mae  = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2   = r2_score(y_val, pred)
    pct5k = (np.abs(y_val - pred) < 5000).mean() * 100

    fold_results.append({"fold": fold, "MAE": mae, "RMSE": rmse, "R2": r2, "<$5k": pct5k})
    print(f"  MAE: ${mae:,.0f}  RMSE: ${rmse:,.0f}  R²: {r2:.4f}  <$5k: {pct5k:.1f}%")

# ── Yhteenveto ────────────────────────────────────────────────────────
res = pd.DataFrame(fold_results)

print(f"\n{'='*60}")
print(f"{'':28} {'MAE':>8} {'RMSE':>8} {'R²':>7} {'<$5k':>7}")
print(f"{'-'*60}")
for _, row in res.iterrows():
    print(f"  Fold {int(row.fold)}               "
          f"${row.MAE:>6,.0f} ${row.RMSE:>6,.0f}  {row.R2:.4f}  {row['<$5k']:.1f}%")
print(f"{'-'*60}")
print(f"  Keskiarvo (CV)       "
      f"${res.MAE.mean():>6,.0f} ${res.RMSE.mean():>6,.0f}  {res.R2.mean():.4f}  {res['<$5k'].mean():.1f}%")
print(f"  Std                  "
      f"${res.MAE.std():>6,.0f} ${res.RMSE.std():>6,.0f}  {res.R2.std():.4f}  {res['<$5k'].std():.1f}%")
print(f"{'-'*60}")
print(f"  Phase 4 test-joukko  $2 461  $4 825  0.9128   89.2%  (vertailu)")
print(f"{'='*60}")

Koko aineisto: 2,605,069 riviä
Featuret: 59

⏳ Fold 1/5 (train: 2,084,055, val: 521,014)...
  MAE: $1,834  RMSE: $2,640  R²: 0.9678  <$5k: 94.2%

⏳ Fold 2/5 (train: 2,084,055, val: 521,014)...
  MAE: $1,837  RMSE: $2,658  R²: 0.9677  <$5k: 94.1%

⏳ Fold 3/5 (train: 2,084,055, val: 521,014)...
  MAE: $1,837  RMSE: $2,644  R²: 0.9680  <$5k: 94.1%

⏳ Fold 4/5 (train: 2,084,055, val: 521,014)...
  MAE: $1,832  RMSE: $2,633  R²: 0.9682  <$5k: 94.3%

⏳ Fold 5/5 (train: 2,084,056, val: 521,013)...
  MAE: $1,833  RMSE: $2,634  R²: 0.9683  <$5k: 94.2%

                                  MAE     RMSE      R²    <$5k
------------------------------------------------------------
  Fold 1               $ 1,834 $ 2,640  0.9678  94.2%
  Fold 2               $ 1,837 $ 2,658  0.9677  94.1%
  Fold 3               $ 1,837 $ 2,644  0.9680  94.1%
  Fold 4               $ 1,832 $ 2,633  0.9682  94.3%
  Fold 5               $ 1,833 $ 2,634  0.9683  94.2%
--------------------------------------------------------

### K-fold tulokset — analyysi

| | MAE | RMSE | R² | <$5k |
|--|----:|-----:|---:|-----:|
| K-fold CV (keskiarvo) | $1 888 | $2 706 | 0.9664 | 93.7% |
| K-fold Std | $3 | $10 | 0.0003 | 0.1% |
| Phase 4 test-joukko | $2 461 | $4 825 | 0.9128 | 89.2% |

---

#### 1. Phase 4:n tulos on luotettava — ei sattumaa

Foldien välinen hajonta on käytännössä nolla (std MAE $3). Tämä tarkoittaa
että malli tuottaa saman tuloksen riippumatta siitä miten data on jaettu.
Phase 4:n yksittäinen jako (random_state=42) ei ollut erityisen hyvä tai
huono — $2 461 on vakaa ja toistettava mittaus.

#### 2. CV-tulos ($1 888) on ylioptimistinen — tietovuoto aggregaateissa

CV yhdisti train- ja test-datat ennen jakoa. Ongelma on että
`brand_model_avg_price`, `county_avg_price` ja muut aggregaattifeaturet
on laskettu koko aineistosta etukäteen — ne sisältävät jo validointifoldin
autojen hintatietoja. Käytännössä malli "näkee" osittain etukäteen mitä
se yrittää ennustaa. Tämä selittää dramaattisen eron CV:n ja Phase 4:n
välillä ($573 MAE). CV-tulosta ei tule käyttää mallin virallisena
suorituskykymittarina.

#### 3. Mitä CV silti kertoi

- **Mallissa ei ole ylisovittumista** — vakaa tulos kaikilla jaoilla
- **Lisädata parantaisi tulosta** — jos aggregaatit laskettaisiin
  ilman vuotoa ja malli koulutettaisiin koko datalla, tulos olisi
  selvästi parempi kuin $2 461
- **Phase 4:n jako on edustava** — ei tarvetta uusia jakoa tai epäillä
  tuloksen sattumanvaraisuutta

#### Johtopäätös

K-fold vahvisti Phase 4:n tuloksen luotettavuuden. $2 461 MAE on
rehellinen arvio mallin kyvystä yleistyä uuteen dataan. CV-tulos
($1 888) kuvaa mallin potentiaalia tilanteessa jossa enemmän dataa
olisi käytettävissä ilman vuoto-ongelmaa — esimerkiksi uudemmalla
aineistolla tehtävässä validoinnissa.

# Varusteflagien kanssa testailut: Ei parannuksia täällä
## Varusteflagit pois — ablation-testi

Testataan vaikuttavatko `opt_`-prefiksillä alkavat varusteflagit 
(navigaatio, kattoluukku, nahkapenkit jne.) tulokseen käytännössä.

**Hypoteesi:** Flagit näkyivät XGBoostin feature importancessa mutta eivät 
dominoineet. CatBoostin tulos muuttunee alle $50 — jos muutos on tätä 
pienempi, varusteflagit ovat käytännössä turhia mallille.

Testi käyttää V2-parametreja (depth=9, l2_leaf_reg=2, iter=800).

In [8]:
# ── Tunnistetaan opt_-featuret ────────────────────────────────────────
opt_cols = [c for c in features if c.startswith("opt_")]
features_no_opt = [c for c in features if not c.startswith("opt_")]
cat_cols_no_opt = [c for c in cat_cols if not c.startswith("opt_")]

print(f"Kaikki featuret:      {len(features)}")
print(f"opt_-featuret:        {len(opt_cols)}")
print(f"Featuret ilman opt_:  {len(features_no_opt)}")
print(f"\nPoistetut sarakkeet:")
for c in opt_cols:
    print(f"  {c}")

# ── Datasettien valmistelu ─────────────────────────────────────────────
X_tr_no = train_df[features_no_opt].copy()
X_te_no = test_df[features_no_opt].copy()

for col in cat_cols_no_opt:
    X_tr_no[col] = X_tr_no[col].astype(str)
    X_te_no[col] = X_te_no[col].astype(str)

# ── V2 ilman opt_-featureja ───────────────────────────────────────────
print("\n⏳ Koulutetaan V2 ilman opt_-featureja...")
v2_no_opt = CatBoostRegressor(
    iterations=800, learning_rate=0.05, depth=9, l2_leaf_reg=2,
    loss_function="RMSE", verbose=False, random_seed=42
)
v2_no_opt.fit(X_tr_no, y_tr, cat_features=cat_cols_no_opt)

m_no_opt = eval_model(v2_no_opt, X_te_no, y_te)
m_v2     = eval_model(v2, X_te, y_te)

# ── Vertailu ─────────────────────────────────────────────────────────
print(f"\n{'Malli':<30} {'MAE':>8} {'RMSE':>8} {'R²':>7} {'<$5k':>7} {'ΔMAE':>9}")
print("-" * 72)

base_mae = m_v2["MAE"]
for name, m in [("V2 (kaikki featuret)", m_v2), ("V2 ilman opt_-flageja", m_no_opt)]:
    delta = f"${m['MAE'] - base_mae:>+7,.0f}" if name != "V2 (kaikki featuret)" else "  —"
    mark  = " ✅" if m["MAE"] < base_mae else (" ❌" if m["MAE"] > base_mae else "")
    print(f"{name:<30} ${m['MAE']:>6,.0f} ${m['RMSE']:>6,.0f}  {m['R2']:.4f}  {m['<$5k']:.1f}%  {delta}{mark}")

print(f"\nMAE-muutos:  ${m_no_opt['MAE'] - m_v2['MAE']:>+,.0f}")
print(f"RMSE-muutos: ${m_no_opt['RMSE'] - m_v2['RMSE']:>+,.0f}")
print(f"R²-muutos:   {m_no_opt['R2'] - m_v2['R2']:>+.4f}")

Kaikki featuret:      59
opt_-featuret:        22
Featuret ilman opt_:  37

Poistetut sarakkeet:
  opt_backup_camera
  opt_bluetooth
  opt_alloy_wheels
  opt_heated_seats
  opt_navigation
  opt_sunroof
  opt_remote_start
  opt_carplay
  opt_android_auto
  opt_leather_seats
  opt_blind_spot
  opt_parking_sensors
  opt_adaptive_cruise
  opt_lane_departure
  opt_cross_traffic
  opt_third_row
  opt_tow_package
  opt_multi_zone
  opt_premium_audio
  opt_cold_weather
  opt_heat_package
  opt_off_road

⏳ Koulutetaan V2 ilman opt_-featureja...

Malli                               MAE     RMSE      R²    <$5k      ΔMAE
------------------------------------------------------------------------
V2 (kaikki featuret)           $ 2,448 $ 4,894  0.9105  89.4%    —
V2 ilman opt_-flageja          $ 2,802 $ 5,269  0.8962  86.2%  $   +354 ❌

MAE-muutos:  $+354
RMSE-muutos: $+375
R²-muutos:   -0.0143


## options_tier — varustemäärän luokittelu

Testataan korvataanko kaikki `opt_`-flagit ja `options_count` yhdellä 
ordinaalisella kategorialla `options_tier` (low / medium / high).

**Motivaatio:** Yksinkertaisempi feature-set, helpompi deployata — 
tarvitaan vain varustelistan kokonaismäärä, ei yksittäisiä flageja.

| options_count | options_tier |
|:---:|:---:|
| ≤ 3 | low |
| 4–7 | medium |
| ≥ 8 | high |

Vertaillaan kolmea vaihtoehtoa rinnakkain:
1. V2 täysi (kaikki featuret)
2. V2 ilman opt_-flageja (options_count mukana)
3. V2 options_tier (opt_ + options_count pois, options_tier tilalle)

In [9]:
# ── Feature-set: opt_ + options_count pois, options_tier tilalle ─────
drop_cols = [c for c in features if c.startswith("opt_")] + ["options_count"]
features_tier = [c for c in features if c not in drop_cols] + ["options_tier"]
cat_cols_tier = [c for c in cat_cols if c not in drop_cols] + ["options_tier"]

print(f"Poistetaan {len(drop_cols)} saraketta (opt_* + options_count)")
print(f"Lisätään: options_tier")
print(f"Feature-setin koko: {len(features)} → {len(features_tier)}")

# ── options_tier luodaan train ja test DataFrameihin ──────────────────
def add_options_tier(df):
    df = df.copy()
    df["options_tier"] = (
        df["options_count"]
        .apply(lambda x: "low" if x <= 3 else ("medium" if x <= 7 else "high"))
        .astype(str)
    )
    return df

train_tier = add_options_tier(train_df)
test_tier  = add_options_tier(test_df)

# Varmistetaan että options_tier on mukana ja opt_/options_count poistettu
X_tr_tier = train_tier[features_tier].copy()
X_te_tier = test_tier[features_tier].copy()

for col in cat_cols_tier:
    X_tr_tier[col] = X_tr_tier[col].astype(str)
    X_te_tier[col] = X_te_tier[col].astype(str)

print(f"\nJakauma options_tier (train):")
print(train_tier["options_tier"].value_counts().sort_index())

# ── Koulutus ──────────────────────────────────────────────────────────
print("\n⏳ Koulutetaan V2 options_tier...")
v2_tier = CatBoostRegressor(
    iterations=800, learning_rate=0.05, depth=9, l2_leaf_reg=2,
    loss_function="RMSE", verbose=False, random_seed=42
)
v2_tier.fit(X_tr_tier, y_tr, cat_features=cat_cols_tier)

m_tier   = eval_model(v2_tier, X_te_tier, y_te)
m_v2     = eval_model(v2, X_te, y_te)
m_no_opt = eval_model(v2_no_opt, X_te_no, y_te)

# ── Kolmen vaihtoehdon vertailu ───────────────────────────────────────
print(f"\n{'Malli':<32} {'MAE':>8} {'RMSE':>8} {'R²':>7} {'<$5k':>7} {'ΔMAE':>9}")
print("-" * 74)

base_mae = m_v2["MAE"]
rows = [
    ("V2 täysi (kaikki featuret)", m_v2),
    ("V2 ilman opt_-flageja",      m_no_opt),
    ("V2 options_tier",            m_tier),
]
for name, m in rows:
    delta = f"${m['MAE'] - base_mae:>+7,.0f}" if name != "V2 täysi (kaikki featuret)" else "  —"
    mark  = " ✅" if m["MAE"] < base_mae else (" ❌" if m["MAE"] > base_mae else "")
    print(f"{name:<32} ${m['MAE']:>6,.0f} ${m['RMSE']:>6,.0f}  {m['R2']:.4f}  {m['<$5k']:.1f}%  {delta}{mark}")

Poistetaan 23 saraketta (opt_* + options_count)
Lisätään: options_tier
Feature-setin koko: 59 → 37

Jakauma options_tier (train):
options_tier
high      656320
low       573516
medium    833664
Name: count, dtype: int64

⏳ Koulutetaan V2 options_tier...

Malli                                 MAE     RMSE      R²    <$5k      ΔMAE
--------------------------------------------------------------------------
V2 täysi (kaikki featuret)       $ 2,448 $ 4,894  0.9105  89.4%    —
V2 ilman opt_-flageja            $ 2,802 $ 5,269  0.8962  86.2%  $   +354 ❌
V2 options_tier                  $ 2,840 $ 5,277  0.8959  85.9%  $   +392 ❌
